# Discovering USGS Water Data with STAC

The [USGS Water Data STAC API](https://api.waterdata.usgs.gov/stac/v0/) is a
catalog for discovering file-based water-data resources. STAC (the SpatioTemporal
Asset Catalog specification) organizes those resources as **Collections**,
**Items**, and downloadable **assets**. At the time of writing, the catalog
contains USGS stage-discharge rating files, but the discovery workflow shown here
also applies as collections are added.

This notebook starts at the catalog landing page, discovers its collections,
searches for a known monitoring location, inspects an Item and its asset, and
walks one page of results at a time.

In [ ]:
from urllib.parse import parse_qs, urlparse

import pandas as pd

from dataretrieval import waterdata

> **Return values.** The generic STAC helpers return a `(document, metadata)`
> tuple. `document` is the unchanged JSON response as a Python dictionary, and
> `metadata` describes the HTTP response. Unlike observation-oriented
> `waterdata` functions, these helpers do not coerce Catalogs, JSON Schemas,
> GeoJSON Items, and their links into one lossy table shape. We create small
> pandas previews below only when a table helps us read part of a document.

## Start at the catalog

A STAC landing page identifies the catalog and advertises the operations it
supports. `waterdata.stac.get_catalog` retrieves that starting document:

In [ ]:
catalog, catalog_metadata = waterdata.stac.get_catalog()

{
    "id": catalog["id"],
    "type": catalog["type"],
    "title": catalog["title"],
    "description": catalog["description"],
    "request_url": catalog_metadata.url,
}

STAC documents are hypermedia documents: their `links` tell a client where to
find collections, conformance declarations, searches, queryables, and API
documentation. Preserving these links is one reason the helpers return raw
documents.

In [ ]:
pd.DataFrame(catalog["links"]).reindex(columns=["rel", "type", "method", "href"])

## Discover collections

A Collection describes a related group of Items. List the available collections
instead of assuming which datasets the catalog currently contains:

In [ ]:
collections_document, _ = waterdata.stac.get_collections()

pd.DataFrame(
    [
        {
            "id": collection["id"],
            "title": collection["title"],
            "description": collection["description"],
        }
        for collection in collections_document["collections"]
    ]
)

Retrieve one Collection to inspect its spatial and temporal extent, license,
providers, and the kinds of assets attached to its Items. The current `ratings`
Collection contains base ratings, corrections, and expanded stage-discharge
tables.

In [ ]:
collection_id = "ratings"
collection, _ = waterdata.stac.get_collection(collection_id)

{
    "id": collection["id"],
    "title": collection["title"],
    "license": collection["license"],
    "extent": collection["extent"],
    "item_assets": collection["item_assets"],
}

### Discover queryable fields

`waterdata.stac.get_queryables` returns a JSON Schema describing fields a server exposes
for filtering. Passing a collection id requests that collection's schema;
omitting it requests the catalog-wide schema. This schema allows additional
properties, so Items may also carry collection-specific fields such as
`monitoring_location_id` and `file_type`.

In [ ]:
queryables, _ = waterdata.stac.get_queryables(collection_id)

pd.DataFrame.from_dict(queryables["properties"], orient="index").reindex(
    columns=["title", "description", "type", "format"]
)

## Search for Items

Suppose we need rating files for monitoring location `USGS-10109000`. A CQL2
text filter selects that location while `collections` limits the search to
ratings. `limit` is the maximum number of Items in this response page, not a
request to flatten every page into one result.

In [ ]:
site = "USGS-10109000"
items_document, search_metadata = waterdata.stac.search(
    collections=[collection_id],
    filter=f"monitoring_location_id = '{site}'",
    filter_lang="cql2-text",
    limit=10,
)

item_preview = []
for feature in items_document["features"]:
    properties = feature["properties"]
    longitude, latitude = feature["geometry"]["coordinates"]
    item_preview.append(
        {
            "id": feature["id"],
            "file_type": properties["file_type"],
            "updated": properties["datetime"],
            "longitude": longitude,
            "latitude": latitude,
        }
    )

print(f"{len(item_preview)} Items from {search_metadata.url}")
pd.DataFrame(item_preview)

The response is a GeoJSON `FeatureCollection`. Each feature is a STAC Item with
an id, geometry, descriptive properties, links, and one or more assets. Use
`waterdata.stac.get_item` when you know the collection and Item ids and want that Item
directly.

In [ ]:
first_feature = items_document["features"][0]
item, item_metadata = waterdata.stac.get_item(collection_id, first_feature["id"])

{
    "id": item["id"],
    "collection": item["collection"],
    "properties": item["properties"],
    "request_url": item_metadata.url,
}

### Follow an Item to its asset

An asset entry describes the file associated with an Item. Its `href` is the
download URL, while `type`, `roles`, size, and description explain what the
file contains. Generic STAC helpers preserve this information but do not decide
how to parse an arbitrary asset.

In [ ]:
pd.DataFrame.from_dict(item["assets"], orient="index").reindex(
    columns=["title", "description", "type", "file:size", "roles", "href"]
)

For rating assets, `get_ratings` is the higher-level convenience function: it
uses this catalog, downloads the selected RDB asset, and parses it into a pandas
DataFrame. Use the generic STAC helpers for discovery and raw Item metadata; use
the specialized helper when you want analysis-ready rating values.

In [ ]:
ratings = waterdata.get_ratings(monitoring_location_id=site, file_type="exsa")
rating = ratings[f"{site}.exsa.rdb"]
rating.head()

## GET and POST searches

`waterdata.stac.search` defaults to GET, which is convenient for simple, shareable query
URLs. Choose POST for structured JSON expressions or geometries that would be
awkward or too long in a URL. Here is the same location filter represented as
CQL2 JSON; POST sends the dictionary as native JSON rather than encoding it into
a query string.

In [ ]:
json_filter = {
    "op": "=",
    "args": [{"property": "monitoring_location_id"}, site],
}
post_document, _ = waterdata.stac.search(
    method="POST",
    collections=[collection_id],
    filter=json_filter,
    filter_lang="cql2-json",
    limit=10,
)

[feature["id"] for feature in post_document["features"]]

## Walk result pages deliberately

Generic STAC calls return one standard response document at a time. If more
Items are available, the response includes a link whose relation is `next`.
Extract its opaque `token` and pass it back as `page_token`; do not construct or
modify the token yourself. This keeps page boundaries and STAC links visible to
the caller.

In [ ]:
first_page, _ = waterdata.stac.search(collections=[collection_id], limit=2)
next_link = next(link for link in first_page["links"] if link["rel"] == "next")
next_token = parse_qs(urlparse(next_link["href"]).query)["token"][0]

second_page, _ = waterdata.stac.search(
    collections=[collection_id],
    limit=2,
    page_token=next_token,
)

pd.DataFrame(
    {
        "first page": [feature["id"] for feature in first_page["features"]],
        "second page": [feature["id"] for feature in second_page["features"]],
    }
)

## Choosing a STAC helper

| Goal | Helper | Document returned |
| --- | --- | --- |
| Discover API operations | `waterdata.stac.get_catalog`, `waterdata.stac.get_conformance` | Catalog or conformance document |
| Discover datasets | `waterdata.stac.get_collections`, `waterdata.stac.get_collection` | Collection list or Collection |
| Browse one Collection | `waterdata.stac.get_items` | GeoJSON ItemCollection page |
| Retrieve a known Item | `waterdata.stac.get_item` | GeoJSON STAC Item |
| Discover filter fields | `waterdata.stac.get_queryables` | JSON Schema |
| Search across Collections | `waterdata.stac.search` | GeoJSON ItemCollection page |
| Download parsed rating tables | `get_ratings` | Dictionary of pandas DataFrames |

All generic helpers preserve the server's standard links. They also use the same
Water Data configuration, API-key handling, retries, and configured base URL as
the rest of `dataretrieval.waterdata`.

## More help

- STAC API: <https://api.waterdata.usgs.gov/stac/v0/>
- STAC specification: <https://stacspec.org/>
- `dataretrieval` API reference: <https://doi-usgs.github.io/dataretrieval-python/reference/waterdata.html>
- See the *USGS Water Data Rating Curve Examples* notebook for a deeper
  `get_ratings` walkthrough.
- Issues / questions: <https://github.com/DOI-USGS/dataretrieval-python/issues>